In [ ]:
# 03b_train_ensemble.ipynb

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import minimize

from src.utils.config import load_config
from src.models.lightgbm_model import LightGBMModel
from src.models.randomforest_model import RandomForestMultiModel
from src.models.ensemble_model import EnsembleModel
from src.models.artifact import save_model_artifact

In [ ]:
# ==========================================
# 1. 환경 설정 및 OOF(Out-of-Fold) 데이터 로드
# ==========================================
cfg = load_config()
ref_date = cfg['project']['reference_date']
model_date = cfg['universe']['model_date']
target_type = cfg['training'].get('target_type', 'log_return')

# 경로 규칙에 따라 모델별 디렉토리 설정
training_base = Path(cfg['paths']['training_dir']) / model_date

print("📥 개별 모델의 검증 폴드 예측 결과 로드 중...")
print("   (앙상블 가중치 최적화용 — 테스트셋과 분리됨)")

# ✅ 변경: predictions.parquet → val_predictions.parquet
df_lgbm_val = pd.read_parquet(training_base / "lightgbm"    / "val_predictions.parquet")
df_rf_val   = pd.read_parquet(training_base / "randomforest" / "val_predictions.parquet")

# ──────────────────────────────────────────────────────────────
# target_cols: 모델이 직접 예측한 원시 타겟 컬럼만 선택
#   log_return 모드: pred_target_log_return_h{n}
#   log_close  모드: pred_target_log_close_h{n}
#
# 역산으로 추가된 pred_log_close_*, pred_close_* 는 제외
# ──────────────────────────────────────────────────────────────
if target_type == "log_return":
    target_prefix = "pred_target_log_return_h"
else:
    target_prefix = "pred_target_log_close_h"

pred_cols = [c for c in df_lgbm_val.columns if c.startswith(target_prefix)]
true_cols = [c.replace("pred_", "true_", 1) for c in pred_cols]

# 검증
missing = [c for c in true_cols if c not in df_lgbm_val.columns]
if missing:
    raise KeyError(f"true_cols에 없는 컬럼: {missing}\n"
                   f"실제 컬럼: {list(df_lgbm_val.columns)}")

print(f"   - target_type : {target_type}")
print(f"   - pred_cols   : {pred_cols}")
print(f"   - true_cols   : {true_cols}")

preds_lgbm = df_lgbm_val[pred_cols].values
preds_rf   = df_rf_val[pred_cols].values
trues      = df_lgbm_val[true_cols].values

# NaN 행 제거
valid_mask = ~np.isnan(trues).any(axis=1)
preds_lgbm = preds_lgbm[valid_mask]
preds_rf   = preds_rf[valid_mask]
trues      = trues[valid_mask]

print(f"   - 유효 샘플 수: {valid_mask.sum():,}")

In [ ]:
# ==========================================
# 2. Scipy 기반 최적 가중치 탐색 (Optimized Blending)
# ==========================================
def blended_rmse(weights):
    w1, w2 = weights[0], 1.0 - weights[0]
    blended = w1 * preds_lgbm + w2 * preds_rf
    return np.sqrt(np.mean((blended - trues) ** 2))

result = minimize(
    blended_rmse,
    x0=[0.8],
    bounds=[(0.0, 1.0)],
    method='L-BFGS-B'
)

w_lgbm = float(result.x[0])
w_rf   = 1.0 - w_lgbm

print(f"\n✅ 최적 가중치 (검증셋 기준)")
print(f"   LightGBM : {w_lgbm:.4f}")
print(f"   RandomForest: {w_rf:.4f}")
print(f"   검증 RMSE: {result.fun:.6f}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# [셀 3] 테스트셋 예측으로 최종 성능 확인 (평가 전용, 가중치 변경 없음)
# ──────────────────────────────────────────────────────────────

print("\n📊 테스트셋 앙상블 성능 확인...")

df_lgbm_test = pd.read_parquet(training_base / "lightgbm"    / "test_predictions.parquet")
df_rf_test   = pd.read_parquet(training_base / "randomforest" / "test_predictions.parquet")

preds_lgbm_test = df_lgbm_test[pred_cols].values
preds_rf_test   = df_rf_test[pred_cols].values
trues_test      = df_lgbm_test[true_cols].values

valid_test_mask = ~np.isnan(trues_test).any(axis=1)
preds_lgbm_test = preds_lgbm_test[valid_test_mask]
preds_rf_test   = preds_rf_test[valid_test_mask]
trues_test      = trues_test[valid_test_mask]

blended_test     = w_lgbm * preds_lgbm_test + w_rf * preds_rf_test
test_rmse_ens    = np.sqrt(np.mean((blended_test      - trues_test) ** 2))
test_rmse_lgbm   = np.sqrt(np.mean((preds_lgbm_test   - trues_test) ** 2))
test_rmse_rf     = np.sqrt(np.mean((preds_rf_test     - trues_test) ** 2))

print(f"   앙상블  RMSE: {test_rmse_ens:.6f}")
print(f"   LGBM    RMSE: {test_rmse_lgbm:.6f}")
print(f"   RF      RMSE: {test_rmse_rf:.6f}")

In [ ]:
# ==========================================
# 4. EnsembleModel 인스턴스화 및 아티팩트 저장
# ==========================================
print("\n📦 학습된 개별 모델 로드 및 조립 중...")
lgbm_path = list((training_base / "lightgbm").glob("*.pkl"))[0]
rf_path = list((training_base / "randomforest").glob("*.pkl"))[0]

lgbm_model = LightGBMModel.load(str(lgbm_path))
rf_model = RandomForestMultiModel.load(str(rf_path))

# 래퍼 모델 조립
ensemble = EnsembleModel(
    model_version=f"v1_ens_{ref_date}",
    models=[lgbm_model, rf_model],
    weights=[w_lgbm, w_rf]
)

# 앙상블 모델 전용 디렉토리에 저장
ensemble_dir = training_base / "ensemble"
ensemble_dir.mkdir(parents=True, exist_ok=True)

save_model_artifact(
    model_name="ensemble",
    model_version=ensemble.model_version,
    model_object=ensemble,
    metadata={
        "weights": {"lightgbm": w_lgbm, "randomforest": w_rf},
        "ensemble_rmse": result.fun,
        "target_columns": ensemble.target_columns
    },
    model_dir=ensemble_dir
)
print(f"✅ 앙상블 모델 저장 완료: {ensemble_dir.name}/")

In [ ]:
print("\n📊 앙상블 검증 결과(OOF) 생성 및 저장 중...")

# 1. 가중치가 적용된 앙상블 예측값 계산 (OOF 데이터 활용)
# preds_lgbm, preds_rf는 앞선 단계에서 로드된 numpy 배열
blended_oof_preds = (w_lgbm * preds_lgbm) + (w_rf * preds_rf)

# 2. 저장용 DataFrame 구축 (df_lgbm의 구조를 복사하여 정답지와 메타데이터 유지)
# valid_mask는 앞서 NaN을 제거할 때 사용한 마스크
df_ensemble_oof = df_lgbm_val[valid_mask].copy()

# 3. 예측값 컬럼 업데이트 (pred_target_log_close_h1 등)
for i, col in enumerate(pred_cols):
    df_ensemble_oof[col] = blended_oof_preds[:, i]

# 4. 앙상블 폴더 내에 predictions.parquet 저장
# 이 파일이 있어야 05_universe_selection.ipynb가 에러 없이 작동합니다.
ensemble_preds_path = ensemble_dir / "predictions.parquet"
df_ensemble_oof.to_parquet(ensemble_preds_path, index=False)

print(f"✅ 앙상블 검증 결과 저장 완료: {ensemble_preds_path}")